In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
import glob

os.chdir('/home/fahad/Documents/MASTERS/CODEBLOCKS/Masters-Thesis-Code/promptMark')

# Load iteration analysis file
df = pd.read_csv('results/iteration-analysis/qwen14_exp3_during_gen_v2_100_mbpp.csv')
df = df.dropna(subset=['iteration_used'])
df['iteration_used'] = df['iteration_used'].astype(int)

print(f"Loaded data shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst rows:")
print(df.head(3))

# 1. ITERATION EFFICIENCY ANALYSIS

In [ ]:
iter_counts = df['iteration_used'].value_counts().sort_index()
iter_pct = (iter_counts / len(df) * 100).round(2)

print("="*60)
print("1. ITERATION EFFICIENCY ANALYSIS")
print("="*60)
print(f"\nTotal tasks: {len(df)}")
print(f"\nIteration Distribution:")
for iter_val, count in iter_counts.items():
    pct = iter_pct[iter_val]
    bar = '█' * int(pct / 2)
    print(f"  {iter_val} iteration(s): {count:3d} tasks ({pct:5.1f}%) {bar}")

print(f"\nIteration Statistics:")
print(f"  Mean: {df['iteration_used'].mean():.2f}")
print(f"  Median: {df['iteration_used'].median():.1f}")
print(f"  Std Dev: {df['iteration_used'].std():.2f}")
print(f"  Min: {df['iteration_used'].min()}, Max: {df['iteration_used'].max()}")

print(f"\nCumulative Completion Rate:")
for i in range(1, 6):
    completed_by_i = (df['iteration_used'] <= i).sum()
    pct = completed_by_i / len(df) * 100
    print(f"  By iteration {i}: {completed_by_i:3d} tasks ({pct:5.1f}%)")

# 2. WATERMARK DETECTION vs ITERATIONS

In [ ]:
print("\n" + "="*60)
print("2. WATERMARK DETECTION vs ITERATIONS")
print("="*60)

watermark_by_iter = df.groupby('iteration_used')['is_watermark'].agg(['sum', 'count', 'mean'])
watermark_by_iter.columns = ['watermarked', 'total', 'detection_rate']
watermark_by_iter['pct'] = watermark_by_iter['detection_rate'] * 100

print(f"\nWatermark Detection by Iteration:")
for iter_val in sorted(df['iteration_used'].unique()):
    row = watermark_by_iter.loc[iter_val]
    print(f"  Iteration {iter_val}: {row['watermarked']:2.0f}/{row['total']:2.0f} detected ({row['pct']:5.1f}%)")

corr = df['iteration_used'].corr(df['is_watermark'])
print(f"\nCorrelation (iterations vs watermark): {corr:.4f}")

watermarked = df[df['is_watermark'] == True]['iteration_used']
not_watermarked = df[df['is_watermark'] == False]['iteration_used']
print(f"\nMean iterations:")
print(f"  Watermarked: {watermarked.mean():.2f}")
print(f"  Non-watermarked: {not_watermarked.mean():.2f}")

# 3. CODE QUALITY METRICS PROGRESSION

In [ ]:
print("\n" + "="*60)
print("3. CODE QUALITY METRICS PROGRESSION")
print("="*60)

print(f"\nCorrectness by Iteration:")
for iter_val in sorted(df['iteration_used'].unique()):
    correct_count = df[df['iteration_used'] == iter_val]['correct'].sum()
    total = (df['iteration_used'] == iter_val).sum()
    pct = correct_count / total * 100
    print(f"  Iteration {iter_val}: {correct_count:2.0f}/{total:2d} correct ({pct:5.1f}%)")

print(f"\nToken Count Evolution:")
tok_by_iter = df.groupby('iteration_used')['token_count'].mean()
for iter_val in sorted(tok_by_iter.index):
    print(f"  Iteration {iter_val}: {tok_by_iter[iter_val]:.1f} tokens")

print(f"\nGreen Count Evolution:")
green_by_iter = df.groupby('iteration_used')['green_count'].mean()
for iter_val in sorted(green_by_iter.index):
    print(f"  Iteration {iter_val}: {green_by_iter[iter_val]:.1f} green tokens")

print(f"\nP-value Evolution:")
pval_by_iter = df.groupby('iteration_used')['p_exact'].mean()
for iter_val in sorted(pval_by_iter.index):
    print(f"  Iteration {iter_val}: {pval_by_iter[iter_val]:.6f}")

# 4. STATISTICAL DISTRIBUTIONS

In [ ]:
print("\n" + "="*60)
print("4. STATISTICAL DISTRIBUTIONS")
print("="*60)

print(f"\nP-value Statistics by Iteration:")
for iter_val in sorted(df['iteration_used'].unique()):
    subset = df[df['iteration_used'] == iter_val]['p_exact']
    print(f"  Iteration {iter_val}:")
    print(f"    Mean={subset.mean():.6f}, Median={subset.median():.6f}, Std={subset.std():.6f}")
    print(f"    Min={subset.min():.6f}, Max={subset.max():.6f}")

print(f"\nToken Count Statistics by Iteration:")
for iter_val in sorted(df['iteration_used'].unique()):
    subset = df[df['iteration_used'] == iter_val]['token_count']
    print(f"  Iteration {iter_val}: Mean={subset.mean():.1f}, Median={subset.median():.1f}, Std={subset.std():.1f}")

print(f"\nGreen Count Statistics by Iteration:")
for iter_val in sorted(df['iteration_used'].unique()):
    subset = df[df['iteration_used'] == iter_val]['green_count']
    print(f"  Iteration {iter_val}: Mean={subset.mean():.1f}, Median={subset.median():.1f}, Std={subset.std():.1f}")

# 5. EFFICIENCY vs QUALITY TRADE-OFFS

In [ ]:
print("\n" + "="*60)
print("6. EFFICIENCY vs QUALITY TRADE-OFFS")
print("="*60)

iter1_df = df[df['iteration_used'] == 1]
iter_max = df['iteration_used'].max()
iter_max_df = df[df['iteration_used'] == iter_max]

print(f"\nEarly vs Late Iterations:")
print(f"  Iteration 1 count: {len(iter1_df)} tasks")
print(f"  Iteration {iter_max} count: {len(iter_max_df)} tasks")

print(f"\nQuality Metrics Comparison:")
print(f"  Metric                 | Iteration 1 | Iteration {iter_max}")
print(f"  {'-'*50}")
print(f"  Avg Token Count        | {iter1_df['token_count'].mean():11.1f} | {iter_max_df['token_count'].mean():11.1f}")
print(f"  Avg Green Count        | {iter1_df['green_count'].mean():11.1f} | {iter_max_df['green_count'].mean():11.1f}")
print(f"  Avg P-value            | {iter1_df['p_exact'].mean():11.6f} | {iter_max_df['p_exact'].mean():11.6f}")
print(f"  Watermark Detected (%) | {iter1_df['is_watermark'].mean()*100:11.1f} | {iter_max_df['is_watermark'].mean()*100:11.1f}")
print(f"  Correct (%)            | {iter1_df['correct'].mean()*100:11.1f} | {iter_max_df['correct'].mean()*100:11.1f}")

early_stop_count = df['stopped_early'].sum()
print(f"\nEarly Stopping:")
print(f"  Tasks with early stop: {early_stop_count}")
print(f"  Percentage: {early_stop_count/len(df)*100:.1f}%")
if df[df['stopped_early']]['iteration_used'].count() > 0:
    print(f"  Avg iterations (early stop): {df[df['stopped_early']]['iteration_used'].mean():.2f}")
    print(f"  Avg iterations (no early stop): {df[~df['stopped_early']]['iteration_used'].mean():.2f}")

# 7. TASK-SPECIFIC PATTERNS

In [ ]:
print("\n" + "="*60)
print("7. TASK-SPECIFIC PATTERNS")
print("="*60)

max_iter = df['iteration_used'].max()
difficult_tasks = df[df['iteration_used'] == max_iter]
easy_tasks = df[df['iteration_used'] == 1]

print(f"\nDifficult Tasks (requiring {int(max_iter)} iterations):")
print(f"  Count: {len(difficult_tasks)}")
print(f"  Success rate: {difficult_tasks['correct'].mean()*100:.1f}%")
print(f"  Watermark rate: {difficult_tasks['is_watermark'].mean()*100:.1f}%")
print(f"  Task IDs: {sorted(difficult_tasks['task_id'].unique())[:10]}")

print(f"\nEasy Tasks (requiring 1 iteration):")
print(f"  Count: {len(easy_tasks)}")
print(f"  Success rate: {easy_tasks['correct'].mean()*100:.1f}%")
print(f"  Watermark rate: {easy_tasks['is_watermark'].mean()*100:.1f}%")
print(f"  Task IDs (first 10): {sorted(easy_tasks['task_id'].unique())[:10]}")

print(f"\nCorrelations:")
corr_correct = df['iteration_used'].corr(df['correct'])
corr_watermark = df['iteration_used'].corr(df['is_watermark'])
corr_pval = df['iteration_used'].corr(df['p_exact'])
print(f"  iterations vs correct: {corr_correct:.4f}")
print(f"  iterations vs watermark: {corr_watermark:.4f}")
print(f"  iterations vs p_exact: {corr_pval:.4f}")

# 8. VISUALIZATIONS

In [ ]:
print("\n" + "="*60)
print("8. COMPREHENSIVE VISUALIZATIONS")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Iteration-Level Analysis: Code Generation & Watermarking Robustness', fontsize=14, fontweight='bold')

# 1. Iteration Distribution
ax1 = axes[0, 0]
iter_dist = df['iteration_used'].value_counts().sort_index()
colors_iter = plt.cm.viridis(np.linspace(0, 1, len(iter_dist)))
ax1.bar(iter_dist.index, iter_dist.values, color=colors_iter, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Iteration Number', fontsize=11, fontweight='bold')
ax1.set_ylabel('Number of Tasks', fontsize=11, fontweight='bold')
ax1.set_title('1. Task Distribution Across Iterations', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)
for i, v in enumerate(iter_dist.values):
    ax1.text(iter_dist.index[i], v + 1, str(int(v)), ha='center', fontweight='bold')

# 2. Watermark Detection Rate
ax2 = axes[0, 1]
watermark_rate_by_iter = []
iter_labels = []
for iter_val in sorted(df['iteration_used'].unique()):
    rate = df[df['iteration_used'] == iter_val]['is_watermark'].mean() * 100
    watermark_rate_by_iter.append(rate)
    iter_labels.append(f"Iter {int(iter_val)}")
ax2.plot(range(len(watermark_rate_by_iter)), watermark_rate_by_iter, marker='o', linewidth=2.5, markersize=8, color='darkblue')
ax2.axhline(y=100, color='red', linestyle='--', alpha=0.5, label='Perfect Detection')
ax2.set_xticks(range(len(watermark_rate_by_iter)))
ax2.set_xticklabels(iter_labels)
ax2.set_ylabel('Watermark Detection Rate (%)', fontsize=11, fontweight='bold')
ax2.set_title('2. Watermark Detection by Iteration', fontsize=12, fontweight='bold')
ax2.set_ylim([90, 102])
ax2.grid(alpha=0.3)
ax2.legend()

# 3. Token Count Evolution
ax3 = axes[0, 2]
token_by_iter = []
for iter_val in sorted(df['iteration_used'].unique()):
    mean_tok = df[df['iteration_used'] == iter_val]['token_count'].mean()
    std_tok = df[df['iteration_used'] == iter_val]['token_count'].std()
    token_by_iter.append((mean_tok, std_tok))
token_means = [x[0] for x in token_by_iter]
token_stds = [x[1] for x in token_by_iter]
ax3.errorbar(range(len(token_means)), token_means, yerr=token_stds, marker='s', linewidth=2.5, markersize=8, 
             color='darkgreen', capsize=5, capthick=2, label='Mean ± Std')
ax3.set_xticks(range(len(iter_labels)))
ax3.set_xticklabels(iter_labels)
ax3.set_ylabel('Average Token Count', fontsize=11, fontweight='bold')
ax3.set_title('3. Token Count Evolution', fontsize=12, fontweight='bold')
ax3.grid(alpha=0.3)
ax3.legend()

# 4. Green Token Distribution
ax4 = axes[1, 0]
green_by_iter = []
for iter_val in sorted(df['iteration_used'].unique()):
    mean_green = df[df['iteration_used'] == iter_val]['green_count'].mean()
    std_green = df[df['iteration_used'] == iter_val]['green_count'].std()
    green_by_iter.append((mean_green, std_green))
green_means = [x[0] for x in green_by_iter]
green_stds = [x[1] for x in green_by_iter]
ax4.bar(range(len(green_means)), green_means, yerr=green_stds, color='darkred', alpha=0.7, capsize=5, edgecolor='black')
ax4.set_xticks(range(len(iter_labels)))
ax4.set_xticklabels(iter_labels)
ax4.set_ylabel('Average Green Token Count', fontsize=11, fontweight='bold')
ax4.set_title('4. Green Token Distribution', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

# 5. Correctness Rate by Iteration
ax5 = axes[1, 1]
correct_rate_by_iter = []
for iter_val in sorted(df['iteration_used'].unique()):
    rate = df[df['iteration_used'] == iter_val]['correct'].mean() * 100
    correct_rate_by_iter.append(rate)
colors_correct = ['green' if x == 100 else 'orange' for x in correct_rate_by_iter]
ax5.bar(range(len(correct_rate_by_iter)), correct_rate_by_iter, color=colors_correct, alpha=0.7, edgecolor='black')
ax5.set_xticks(range(len(iter_labels)))
ax5.set_xticklabels(iter_labels)
ax5.set_ylabel('Correctness Rate (%)', fontsize=11, fontweight='bold')
ax5.set_title('5. Code Correctness by Iteration', fontsize=12, fontweight='bold')
ax5.set_ylim([0, 105])
ax5.grid(axis='y', alpha=0.3)
for i, v in enumerate(correct_rate_by_iter):
    ax5.text(i, v + 2, f'{v:.0f}%', ha='center', fontweight='bold')

# 6. P-value Trends
ax6 = axes[1, 2]
pval_by_iter_log = []
for iter_val in sorted(df['iteration_used'].unique()):
    pval = df[df['iteration_used'] == iter_val]['p_exact'].mean()
    pval_by_iter_log.append(max(pval, 1e-8))  # Avoid log(0)
ax6.semilogy(range(len(pval_by_iter_log)), pval_by_iter_log, marker='d', linewidth=2.5, markersize=8, color='purple')
ax6.set_xticks(range(len(iter_labels)))
ax6.set_xticklabels(iter_labels)
ax6.set_ylabel('Average P-value (log scale)', fontsize=11, fontweight='bold')
ax6.set_title('6. Statistical Significance (P-values)', fontsize=12, fontweight='bold')
ax6.grid(alpha=0.3, which='both')

plt.tight_layout()
plt.savefig('output/iteration_analysis.png', dpi=300, bbox_inches='tight')
print("\n✓ Visualization saved to output/iteration_analysis.png")
plt.show()

# Export summary statistics to JSON
import json

summary_stats = {
    "total_tasks": len(df),
    "iterations_used": {
        str(int(i)): {
            "count": int((df['iteration_used'] == i).sum()),
            "percentage": float((df['iteration_used'] == i).sum() / len(df) * 100),
            "avg_tokens": float(df[df['iteration_used'] == i]['token_count'].mean()),
            "avg_green_tokens": float(df[df['iteration_used'] == i]['green_count'].mean()),
            "correctness_rate": float(df[df['iteration_used'] == i]['correct'].mean() * 100),
            "watermark_detection_rate": float(df[df['iteration_used'] == i]['is_watermark'].mean() * 100),
            "avg_p_exact": float(df[df['iteration_used'] == i]['p_exact'].mean())
        }
        for i in sorted(df['iteration_used'].unique())
    },
    "overall_stats": {
        "mean_iterations": float(df['iteration_used'].mean()),
        "median_iterations": float(df['iteration_used'].median()),
        "std_iterations": float(df['iteration_used'].std()),
        "overall_correctness": float(df['correct'].mean() * 100),
        "overall_watermark_rate": float(df['is_watermark'].mean() * 100),
        "early_stopping_rate": float(df['stopped_early'].mean() * 100)
    },
    "correlations": {
        "iterations_vs_correct": float(df['iteration_used'].corr(df['correct'])),
        "iterations_vs_watermark": float(df['iteration_used'].corr(df['is_watermark'])),
        "iterations_vs_pvalue": float(df['iteration_used'].corr(df['p_exact'])),
        "token_count_vs_correct": float(df['token_count'].corr(df['correct'])),
        "green_tokens_vs_correct": float(df['green_count'].corr(df['correct']))
    }
}

with open('output/iteration_analysis_summary.json', 'w') as f:
    json.dump(summary_stats, f, indent=2)
print("✓ Summary statistics exported to output/iteration_analysis_summary.json")

# Export detailed results to CSV
output_csv = df.copy()
output_csv = output_csv.sort_values(['iteration_used', 'task_id'])
output_csv.to_csv('output/iteration_analysis_detailed.csv', index=False)
print("✓ Detailed results exported to output/iteration_analysis_detailed.csv")

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)
print(f"\nKey Findings:")
print(f"  • Most tasks completed in first iteration: {(df['iteration_used'] == 1).sum()}/{len(df)} ({(df['iteration_used'] == 1).sum()/len(df)*100:.1f}%)")
print(f"  • Overall watermark detection rate: {df['is_watermark'].mean()*100:.1f}%")
print(f"  • Overall correctness rate: {df['correct'].mean()*100:.1f}%")
print(f"  • Early stopping percentage: {df['stopped_early'].mean()*100:.1f}%")